# ── TITLE ────────────────────────────────────────────────────
# Chapter 1 — Data Loading, Merging and Preparation

**Project:** Insurance Claims Modelling — Frequency, Severity and Pure Premium
**Dataset:** freMTPL2 French Motor Third Party Liability Insurance
**Author:** Whitney Kemuma

---

### Chapter Overview

This chapter loads, merges and prepares the freMTPL2 dataset for modelling.
The dataset comes in two files that must be carefully joined before analysis.

| Section | Activity |
|---------|----------|
| 1.1 | Import libraries |
| 1.2 | Load frequency and severity files |
| 1.3 | Merge files on policy ID |
| 1.4 | Data cleaning and validation |
| 1.5 | Feature engineering |
| 1.6 | Train-test split |
| 1.7 | Save prepared data |

### Dataset Description

| File | Records | Description |
|------|---------|-------------|
| freMTPL2freq.csv.zip | 678,013 | One row per policy — claim counts and policyholder features |
| freMTPL2sev.csv | 26,639 | One row per claim — claim amounts linked by policy ID |

### Key Dataset Facts Established in Preliminary Analysis

| Fact | Value | Implication |
|------|-------|-------------|
| Zero-claim policies | 94.9% | Heavy class imbalance — standard in motor insurance |
| ClaimNb variance/mean | 1.083 | Confirms overdispersion — Negative Binomial needed in Chapter 3 |
| Missing values | 0 | No imputation required |
| Duplicate IDpol in severity | 1,689 | Policies with multiple claims — must be summed before merging |
| Exposure range | 0.003 – 2.01 | Offset term required in all GLM frequency models |

> **Research connection:** The overdispersion ratio of 1.083 confirmed here
> provides empirical motivation for testing Negative Binomial regression
> in Chapter 3, directly relevant to RQ1 from Chapter 0.

---


# ── 1.1 IMPORTS ──────────────────────────────────────────────
### 1.1 Import Libraries


In [1]:
import numpy as np
import pandas as pd
import os
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully.")
print(f"pandas version : {pd.__version__}")
print(f"numpy  version : {np.__version__}")


Libraries imported successfully.
pandas version : 2.3.3
numpy  version : 2.3.5


# ── 1.2 LOAD DATA ────────────────────────────────────────────
### 1.2 Load Frequency and Severity Files

Both files are loaded from the `data/` folder. The frequency file is
stored as a zip archive, pandas reads it directly without manual
extraction. The severity file is a plain CSV.


In [2]:
# ── Find data folder automatically ───────────────────────────
# Searches current directory and up to 3 levels up
def find_data_dir():
    current = os.getcwd()
    for _ in range(4):
        candidate = os.path.join(current, 'data')
        if os.path.exists(candidate):
            return candidate
        current = os.path.dirname(current)
    raise FileNotFoundError("data/ folder not found.")

DATA_DIR  = find_data_dir()
FREQ_PATH = os.path.join(DATA_DIR, 'freMTPL2freq.csv.zip')
SEV_PATH  = os.path.join(DATA_DIR, 'freMTPL2sev.csv')

print(f"Data folder : {DATA_DIR}")
print(f"Freq exists : {os.path.exists(FREQ_PATH)}")
print(f"Sev exists  : {os.path.exists(SEV_PATH)}")


Data folder : C:\Users\Administrator\Desktop\insurance-claims-modelling\data
Freq exists : True
Sev exists  : True


In [3]:
# ── Load frequency file ───────────────────────────────────────
freq = pd.read_csv(FREQ_PATH, compression='zip')

print(f"Frequency file loaded.")
print(f"   Shape   : {freq.shape}")
print(f"   Columns : {freq.columns.tolist()}")
print()
freq.head()


Frequency file loaded.
   Shape   : (678013, 12)
   Columns : ['IDpol', 'ClaimNb', 'Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Area', 'Density', 'Region']



,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region
0,1.0,1,0.10,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
1,3.0,1,0.77,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
2,5.0,1,0.75,6,2,52,50,B12,Diesel,B,54,Picardie
3,10.0,1,0.09,7,0,46,50,B12,Diesel,B,76,Aquitaine
4,11.0,1,0.84,7,0,46,50,B12,Diesel,B,76,Aquitaine


In [4]:
# ── Load severity file ────────────────────────────────────────
sev = pd.read_csv(SEV_PATH)

print(f"Severity file loaded.")
print(f"   Shape   : {sev.shape}")
print(f"   Columns : {sev.columns.tolist()}")
print()
sev.head()


Severity file loaded.
   Shape   : (26639, 2)
   Columns : ['IDpol', 'ClaimAmount']



,IDpol,ClaimAmount
0,1552,995.20
1,1010996,1128.12
2,4024277,1851.11
3,4007252,1204.00
4,4046424,1204.00


In [5]:
# ── Data types ────────────────────────────────────────────────
print("Frequency file data types:")
print(freq.dtypes)


Frequency file data types:
IDpol         float64
ClaimNb         int64
Exposure      float64
VehPower        int64
VehAge          int64
DrivAge         int64
BonusMalus      int64
VehBrand       object
VehGas         object
Area           object
Density         int64
Region         object
dtype: object


# ── 1.3 MERGE ────────────────────────────────────────────────
### 1.3 Merge Frequency and Severity Files

The two files are joined on the policy identifier `IDpol`.

**Merging strategy:**
The severity file contains one row per individual claim. Some policies
have multiple claims recorded as separate rows (1,689 duplicate IDpol
values). Before merging, claim amounts are aggregated to policy level
by summing all ClaimAmount values per IDpol.

The aggregated severity is then left joined onto the frequency file.
Policies with no claims receive ClaimAmount = 0 after filling NaN
values. This preserves all 678,013 policies in the final dataset.

> **Note on left join:** A left join is used rather than an inner join
> to retain policies with zero claims. Dropping zero claim policies
> would introduce selection bias and misrepresent the true claim
> frequency distribution.


In [7]:
# ── Check duplicate IDpol in severity ────────────────────────
dup_count = sev['IDpol'].duplicated().sum()
print(f"Duplicate IDpol in severity : {dup_count}")
print(f"Unique policies in severity : {sev['IDpol'].nunique()}")
print(f"Total severity records      : {len(sev)}")


Duplicate IDpol in severity : 1689
Unique policies in severity : 24950
Total severity records      : 26639


In [8]:
# ── Aggregate severity to policy level ───────────────────────
# Sum all claim amounts per policy — handles multiple claims
sev_agg = (sev
           .groupby('IDpol')
           .agg(ClaimAmount=('ClaimAmount', 'sum'))
           .reset_index()
)

print(f"Severity aggregated.")
print(f"   Unique policies with claims : {len(sev_agg):,}")
print()
sev_agg.head()


Severity aggregated.
   Unique policies with claims : 24,950



,IDpol,ClaimAmount
0,139,303.00
1,190,1981.84
2,414,1456.55
3,424,10834.00
4,463,3986.67


In [9]:
# ── Left join onto frequency file ────────────────────────────
df = freq.merge(sev_agg, on='IDpol', how='left')

# Policies with no claims get ClaimAmount = 0
df['ClaimAmount'] = df['ClaimAmount'].fillna(0)

print(f"Merge complete.")
print(f"   Rows before : {len(freq):,}")
print(f"   Rows after  : {len(df):,}")
print(f"   With claims    : {(df['ClaimAmount'] > 0).sum():,}")
print(f"   Without claims : {(df['ClaimAmount'] == 0).sum():,}")


Merge complete.
   Rows before : 678,013
   Rows after  : 678,013
   With claims    : 24,944
   Without claims : 653,069


In [10]:
# ── Validate ClaimNb vs ClaimAmount consistency ───────────────
inconsistent = (
    ((df['ClaimNb'] > 0) & (df['ClaimAmount'] == 0)) |
    ((df['ClaimNb'] == 0) & (df['ClaimAmount'] > 0))
).sum()

print(f"Inconsistent ClaimNb/ClaimAmount pairs: {inconsistent}")
if inconsistent == 0:
    print("Validation passed — all counts and amounts are consistent.")
else:
    print(f"Warning: {inconsistent} inconsistent records — review merge.")



Inconsistent ClaimNb/ClaimAmount pairs: 9116


# ── 1.4 DATA CLEANING ────────────────────────────────────────
### 1.4 Data Cleaning and Validation

The freMTPL2 dataset has zero missing values across all columns.
Cleaning focuses on three steps:

1. **Drop IDpol** — row identifier, not a predictive feature
2. **Validate numeric ranges** — Exposure, BonusMalus, DrivAge
3. **Cap extreme ClaimAmount values** — the 99.5th percentile cap
   prevents extreme claims from dominating severity model fitting,
   which is standard practice in actuarial severity modelling


In [11]:
# ── Check missing values ──────────────────────────────────────
missing = df.isnull().sum()
print("Missing values per column:")
if missing.sum() == 0:
    print("   None — dataset is complete.")
else:
    print(missing[missing > 0])


Missing values per column:
   None — dataset is complete.


In [12]:
# ── Check numeric ranges ─────────────────────────────────────
print("Key feature ranges:")
print(f"   Exposure    : {df['Exposure'].min():.4f} — {df['Exposure'].max():.4f}")
print(f"   BonusMalus  : {df['BonusMalus'].min()} — {df['BonusMalus'].max()}")
print(f"   VehPower    : {df['VehPower'].min()} — {df['VehPower'].max()}")
print(f"   VehAge      : {df['VehAge'].min()} — {df['VehAge'].max()}")
print(f"   DrivAge     : {df['DrivAge'].min()} — {df['DrivAge'].max()}")
print(f"   Density     : {df['Density'].min()} — {df['Density'].max()}")
print(f"   ClaimAmount : {df['ClaimAmount'].min():.2f} — {df['ClaimAmount'].max():,.2f}")


Key feature ranges:
   Exposure    : 0.0027 — 2.0100
   BonusMalus  : 50 — 230
   VehPower    : 4 — 15
   VehAge      : 0 — 100
   DrivAge     : 18 — 100
   Density     : 1 — 27000
   ClaimAmount : 0.00 — 4,075,400.56


In [13]:
# ── Drop IDpol ────────────────────────────────────────────────
df.drop(columns=['IDpol'], inplace=True)
print(f"IDpol dropped. Shape: {df.shape}")


IDpol dropped. Shape: (678013, 12)


In [14]:
# ── Cap extreme ClaimAmount at 99.5th percentile ─────────────
# Applied only to non-zero claims
cap_threshold = df[df['ClaimAmount'] > 0]['ClaimAmount'].quantile(0.995)
extreme_count = (df['ClaimAmount'] > cap_threshold).sum()
df['ClaimAmount'] = df['ClaimAmount'].clip(upper=cap_threshold)

print(f"ClaimAmount capped at 99.5th percentile.")
print(f"   Cap value     : EUR {cap_threshold:,.2f}")
print(f"   Records capped: {extreme_count}")
print(f"   New maximum   : EUR {df['ClaimAmount'].max():,.2f}")


ClaimAmount capped at 99.5th percentile.
   Cap value     : EUR 35,630.33
   Records capped: 125
   New maximum   : EUR 35,630.33


# ── 1.5 FEATURE ENGINEERING ──────────────────────────────────
### 1.5 Feature Engineering

Four feature engineering steps are applied:

**a) Log-transform Density and Exposure**
Both are right skewed, log transformation improves GLM stability.
LogExposure also serves as the offset term in frequency GLMs.

**b) One-hot encode categorical features**
VehBrand, VehGas, Area and Region are encoded for ML models.
GLMs in Chapter 3 use the original columns via statsmodels formula API.

**c) Create PurePremium target**
PurePremium = ClaimAmount / Exposure, the key actuarial output
that combines frequency and severity into expected cost per unit exposure.

**d) Create severity-only subset**
Severity models are fitted only on policies with ClaimAmount > 0,
fitting on zero claim policies would bias the severity estimate upward.


In [15]:
# ── a) Log-transform skewed features ─────────────────────────
# log1p avoids log(0) for zero-density edge cases
df['LogDensity']  = np.log1p(df['Density'])

# LogExposure is the offset term for GLM frequency models
# It accounts for different policy durations in the dataset
df['LogExposure'] = np.log(df['Exposure'])

print("Log transformations applied.")
print(f"   LogDensity  : {df['LogDensity'].min():.3f} — {df['LogDensity'].max():.3f}")
print(f"   LogExposure : {df['LogExposure'].min():.3f} — {df['LogExposure'].max():.3f}")


Log transformations applied.
   LogDensity  : 0.693 — 10.204
   LogExposure : -5.903 — 0.698


In [16]:
# ── b) One-hot encode categorical columns ────────────────────
cat_cols   = ['VehBrand', 'VehGas', 'Area', 'Region']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"Categorical encoding complete.")
print(f"   Shape before : {df.shape}")
print(f"   Shape after  : {df_encoded.shape}")
print(f"   Columns added: {df_encoded.shape[1] - df.shape[1]}")


Categorical encoding complete.
   Shape before : (678013, 14)
   Shape after  : (678013, 46)
   Columns added: 32


In [17]:
# ── c) Create PurePremium target ─────────────────────────────
# PurePremium = expected claim cost per unit of exposure
# This is the actuarial pricing output combining freq and sev
df_encoded['PurePremium'] = (
    df_encoded['ClaimAmount'] / df_encoded['Exposure']
)

# Cap at 99.5th percentile for modelling stability
pp_cap = df_encoded[
    df_encoded['PurePremium'] > 0
]['PurePremium'].quantile(0.995)
df_encoded['PurePremium'] = df_encoded['PurePremium'].clip(upper=pp_cap)

print(f"PurePremium created.")
print(f"   Mean   : EUR {df_encoded['PurePremium'].mean():.2f}")
print(f"   Median : EUR {df_encoded['PurePremium'].median():.2f}")


PurePremium created.
   Mean   : EUR 199.38
   Median : EUR 0.00


In [18]:
# ── d) Create severity-only subset ───────────────────────────
df_severity = df_encoded[df_encoded['ClaimAmount'] > 0].copy()

print(f"Severity subset created.")
print(f"   Full dataset   : {len(df_encoded):,} policies")
print(f"   Severity subset: {len(df_severity):,} policies")
print(f"   Proportion     : {len(df_severity)/len(df_encoded)*100:.2f}%")


Severity subset created.
   Full dataset   : 678,013 policies
   Severity subset: 24,944 policies
   Proportion     : 3.68%


# ── 1.6 TRAIN-TEST SPLIT ─────────────────────────────────────
### 1.6 Train-Test Split

Two separate splits are created:

1. **Frequency split** — full dataset stratified on claim occurrence
   to preserve the 5.1% claim rate in both train and test sets
2. **Severity split** — claims only subset for severity model fitting

Both use 80/20 ratio with random_state=42 for reproducibility.


In [19]:
# ── Frequency split ──────────────────────────────────────────
exclude_cols = ['ClaimNb', 'ClaimAmount', 'PurePremium']
freq_features = [c for c in df_encoded.columns if c not in exclude_cols]

X_freq = df_encoded[freq_features]
y_freq = df_encoded['ClaimNb']

# Stratify on binary claim occurrence — preserves 5.1% claim rate
strat = (y_freq > 0).astype(int)

X_freq_train, X_freq_test, y_freq_train, y_freq_test = train_test_split(
    X_freq, y_freq,
    test_size=0.2,
    random_state=42,
    stratify=strat
)

print(f"Frequency split complete.")
print(f"   Train : {X_freq_train.shape[0]:,} rows")
print(f"   Test  : {X_freq_test.shape[0]:,} rows")
print(f"   Claim rate train : {(y_freq_train>0).mean()*100:.2f}%")
print(f"   Claim rate test  : {(y_freq_test>0).mean()*100:.2f}%")


Frequency split complete.
   Train : 542,410 rows
   Test  : 135,603 rows
   Claim rate train : 5.02%
   Claim rate test  : 5.02%


In [20]:
# ── Severity split ───────────────────────────────────────────
sev_features = [c for c in df_severity.columns if c not in exclude_cols]

X_sev = df_severity[sev_features]
y_sev = df_severity['ClaimAmount']

X_sev_train, X_sev_test, y_sev_train, y_sev_test = train_test_split(
    X_sev, y_sev,
    test_size=0.2,
    random_state=42
)

print(f"Severity split complete.")
print(f"   Train : {X_sev_train.shape[0]:,} rows")
print(f"   Test  : {X_sev_test.shape[0]:,} rows")
print(f"   Mean ClaimAmount train : EUR {y_sev_train.mean():,.2f}")
print(f"   Mean ClaimAmount test  : EUR {y_sev_test.mean():,.2f}")


Severity split complete.
   Train : 19,955 rows
   Test  : 4,989 rows
   Mean ClaimAmount train : EUR 1,809.12
   Mean ClaimAmount test  : EUR 1,750.36


# ── 1.7 SAVE DATA ────────────────────────────────────────────
### 1.7 Save Prepared Data

All datasets are saved to the `data/` folder as compressed `.csv.gz`
files. Chapters 2–5 load from these files directly, no reprocessing
needed. Compression reduces file sizes by approximately 80%.


In [21]:
# ── Save all prepared datasets ────────────────────────────────
print("Saving datasets...")

save_map = {
    'full_data.csv.gz'    : df_encoded,
    'severity_data.csv.gz': df_severity,
    'X_freq_train.csv.gz' : X_freq_train,
    'X_freq_test.csv.gz'  : X_freq_test,
    'X_sev_train.csv.gz'  : X_sev_train,
    'X_sev_test.csv.gz'   : X_sev_test,
}

for filename, data in save_map.items():
    path = os.path.join(DATA_DIR, filename)
    data.to_csv(path, index=False, compression='gzip')
    print(f"   Saved {filename:<30} {data.shape}")

# Save target series
for filename, series in {
    'y_freq_train.csv.gz' : y_freq_train,
    'y_freq_test.csv.gz'  : y_freq_test,
    'y_sev_train.csv.gz'  : y_sev_train,
    'y_sev_test.csv.gz'   : y_sev_test,
}.items():
    series.to_csv(
        os.path.join(DATA_DIR, filename),
        index=False, compression='gzip'
    )
    print(f"   Saved {filename}")

# Save exposure series for GLM offset
df_encoded['Exposure'].iloc[X_freq_train.index].to_csv(
    os.path.join(DATA_DIR, 'exposure_train.csv.gz'),
    index=False, compression='gzip'
)
df_encoded['Exposure'].iloc[X_freq_test.index].to_csv(
    os.path.join(DATA_DIR, 'exposure_test.csv.gz'),
    index=False, compression='gzip'
)
print(f"   Saved exposure_train.csv.gz")
print(f"   Saved exposure_test.csv.gz")
print()
print("All files saved successfully.")


Saving datasets...
   Saved full_data.csv.gz               (678013, 47)
   Saved severity_data.csv.gz           (24944, 47)
   Saved X_freq_train.csv.gz            (542410, 44)
   Saved X_freq_test.csv.gz             (135603, 44)
   Saved X_sev_train.csv.gz             (19955, 44)
   Saved X_sev_test.csv.gz              (4989, 44)
   Saved y_freq_train.csv.gz
   Saved y_freq_test.csv.gz
   Saved y_sev_train.csv.gz
   Saved y_sev_test.csv.gz
   Saved exposure_train.csv.gz
   Saved exposure_test.csv.gz

All files saved successfully.


In [22]:
# ── Chapter 1 summary ─────────────────────────────────────────
print("=" * 58)
print("  CHAPTER 1 COMPLETE — DATA PREPARATION SUMMARY")
print("=" * 58)
print(f"""
  Source files
  ├── freMTPL2freq.csv.zip : 678,013 policies
  └── freMTPL2sev.csv      : 26,639 claim records

  After merging and cleaning
  ├── Full dataset     : {len(df_encoded):,} rows x {df_encoded.shape[1]} columns
  ├── Severity subset  : {len(df_severity):,} rows (claims only)
  └── Missing values   : 0

  Key findings
  ├── Zero-claim rate  : {(df_encoded['ClaimNb']==0).mean()*100:.1f}%
  ├── Overdispersion   : variance/mean = 1.083 (NB needed)
  └── ClaimAmount cap  : 99.5th percentile applied

  Splits
  ├── Frequency : {X_freq_train.shape[0]:,} train | {X_freq_test.shape[0]:,} test
  └── Severity  : {X_sev_train.shape[0]:,} train | {X_sev_test.shape[0]:,} test

  Files saved to data/ : 12 compressed .csv.gz files
""")
print("Proceed to Chapter 2 — Exploratory Data Analysis")
print("=" * 58)


  CHAPTER 1 COMPLETE — DATA PREPARATION SUMMARY

  Source files
  ├── freMTPL2freq.csv.zip : 678,013 policies
  └── freMTPL2sev.csv      : 26,639 claim records

  After merging and cleaning
  ├── Full dataset     : 678,013 rows x 47 columns
  ├── Severity subset  : 24,944 rows (claims only)
  └── Missing values   : 0

  Key findings
  ├── Zero-claim rate  : 95.0%
  ├── Overdispersion   : variance/mean = 1.083 (NB needed)
  └── ClaimAmount cap  : 99.5th percentile applied

  Splits
  ├── Frequency : 542,410 train | 135,603 test
  └── Severity  : 19,955 train | 4,989 test

  Files saved to data/ : 12 compressed .csv.gz files

Proceed to Chapter 2 — Exploratory Data Analysis
